# Irrigation Training — **v2.21c** (additive terminal-yield)

Separate version (prefix `td3_v221c_*`, manifest 2.21c). **v2.21c = v2.20 Run A (exact n-step + the dense INCREMENT biomass reward, the best baseline) + an ADDITIVE terminal-yield term.**

Unlike the gamma-shaping runs (v2.21 / v2.21-v2), which *replaced* r1 and gutted the dense signal (under-irrigation, worse drought/waterlog), this **keeps the full increment r1** and simply adds `alpha_T * x4_final/X4_REF` once at episode end — lifting the weight on final yield (endpoint coefficient gamma^T → gamma^T(1+alpha_T), ~0.39 → ~0.78 at alpha_T=1.0) toward MPC's terminal-biomass objective, without touching gamma. 

**The shared `gym_env.py` also reverts** X4_REF 900→600 and ALPHA2 0.01→0.016 (the v2.20 Run A values; the 900/0.01 experiment was worse and broke the MPC α2 mirror).

**Before running:** push the modified `gym_env.py`, plus `configs_v221c.py` and `train_v221c_td3.py`; or run the *Write v2.21c files* cell. ~1–1.5 hr T4.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v220_td3_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:',DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0). Same stack as the SAC/TD3 runs.
import subprocess, sys, os
WORK='/content'; repo=os.path.join(WORK,'thesis')
if os.path.exists(repo): subprocess.run(['rm','-rf',repo],check=True)
subprocess.run(['git','clone','https://github.com/taratorbati/thesis.git',repo],check=True)
os.chdir(repo); sys.path.insert(0,repo)
subprocess.run(['pip','install','--quiet','stable-baselines3==2.6.0','gymnasium','wandb','pytest'],check=True)
import torch; print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


## [OPTIONAL] Write v2.21c files
Skip if already pushed. Writes the modified `gym_env.py` (reverts X4_REF→600, ALPHA2→0.016, adds the `reward_terminal_yield` param), `configs_v221c.py`, and `train_v221c_td3.py`.

In [ ]:
# [OPTIONAL] Write the v2.21c files into the cloned repo. Skip if already pushed.
from pathlib import Path
REPO = '/content/thesis'
_files = {
    'src/rl/configs_v221c.py':
        '23207372632f726c2f636f6e666967735f76323231632e7079202076322e3231630a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d0a232076322e323163203d2076322e32302052756e204120286578616374206e2d73746570'
        '202b207468652064656e736520494e4352454d454e542062696f6d617373207265776172642c2074686520626573740a2320'
        '626173656c696e6529202b20616e204144444954495645207465726d696e616c2d7969656c64207465726d2e204f6e652076'
        '61726961626c65206f6e20746f70206f66207468652070726f76656e2072756e2e0a230a2320576879206164646974697665'
        '20286e6f74207468652067616d6d612d73686170696e6720666f726d293a0a232020202d2076322e32302052756e20412028'
        '696e6372656d656e7420723129207761732074686520626573743a2039392e3825204d5043207969656c642c206265617420'
        '4d5043206f6e2077617465726c6f672e0a232020202d205468652067616d6d612d636f72726563742073686170696e672028'
        '76322e3231202f2076322e32312d763229205245504c4143454420723120616e64206d616465207468696e677320776f7273'
        '653a0a232020202020697420677574746564207468652064656e73652062696f6d617373207369676e616c2c20736f207468'
        '6520706f6c69637920756e6465722d69727269676174656420616e64206d69732d616c6c6f63617465642e0a232020202d20'
        '546865206164646974697665207465726d696e616c207465726d204b45455053207468652066756c6c20696e6372656d656e'
        '74207369676e616c202862696f6d6173735f73686170696e673d46616c7365290a232020202020616e642073696d706c7920'
        '4144445320616c7068615f54202a2078345f66696e616c2f58345f524546206f6e636520617420657069736f646520656e64'
        '2e204974206c69667473207468650a232020202020656e64706f696e74207765696768742066726f6d2067616d6d615e5420'
        '287e302e33392920746f2067616d6d615e542a28312b616c7068615f542920287e302e373820617420616c7068615f543d31'
        '2e30290a2320202020202d2d2070756c6c696e6720746865206f626a65637469766520746f77617264204d50432773207465'
        '726d696e616c2d62696f6d61737320636f7374202d2d20776974686f75742072656d6f76696e670a23202020202074686520'
        '64656e7365207369676e616c2e2067616d6d6120697320756e746f75636865642c20736f206e2d737465702073746162696c'
        '697479206973207072657365727665642e0a230a23205363616c65733a2067796d5f656e762e707920726576657274732058'
        '345f524546203930302d3e36303020616e6420414c5048413220302e30312d3e302e30313620287468652076322e32302052'
        '756e20410a232076616c756573293b2074686520393030202f20302e3031206578706572696d656e742077617320776f7273'
        '6520616e642062726f6b6520746865204d504320616c70686132206d6972726f722e0a2320616c7068615f54203d20312e30'
        '20697320736561736f6e2d73756d2d63616c6962726174656420287465726d696e616c20626f6e7573207e312e302a78345f'
        '66696e616c2f363030207e3d20312e3337207e3d0a23207468652063756d756c61746976652064656e73652d723120736561'
        '736f6e2d73756d207e312e32372c20746865206d6574686f64207573656420666f7220414c504841365f4c494e292e205475'
        '6e6520696e0a23205b302e352c20312e355d3a206c6f776572206966207765742d796561722077617465726c6f6720726567'
        '726573736573206f7220715f7072656420646573746162696c697365733b206869676865722069660a23207468652064726f'
        '756768742073656573617720646f6573206e6f74206d6f76652e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a47414d4d415f42'
        '415345203d20302e39390a0a52554e5f41203d2064696374280a202020206c6162656c3d227465726d7969656c64222c0a20'
        '20202023202d2d2d20696e686572697465642066726f6d2076322e32302052756e204120287468652065786163742d6e2d73'
        '7465702073746162696c697365722c20756e6368616e67656429202d2d2d0a202020206e5f73746570733d352c0a20202020'
        '67616d6d615f626173653d47414d4d415f424153452c0a202020206c6561726e696e675f7374617274733d35305f3030302c'
        '0a202020207265776172645f64755f616c7068613d302e3030352c0a20202020706f6c6963795f64656c61793d322c0a2020'
        '20207461726765745f706f6c6963795f6e6f6973653d302e322c0a202020207461726765745f6e6f6973655f636c69703d30'
        '2e352c0a202020206163746f725f6c725f6d756c743d312e302c0a202020206163746f725f7761726d75705f757064617465'
        '733d302c0a202020206578706f73655f707265765f753d46616c73652c0a2020202023202d2d2d2076322e323163202d2d2d'
        '0a2020202062696f6d6173735f73686170696e673d46616c73652c2020202020202023204b454550207468652064656e7365'
        '20696e6372656d656e74207231202876322e323020666f726d29202d2d20646f204e4f542067616d6d612d73686170650a20'
        '2020207265776172645f7465726d696e616c5f7969656c643d312e302c2020202320746865204f4e4c59206368616e67653a'
        '204144444954495645207465726d696e616c2d7969656c6420626f6e75730a290a0a434f4e46494753203d207b2241223a20'
        '52554e5f417d0a'
        ,
    'src/rl/train_v221c_td3.py':
        '...'
        '696d6573746570732c0a20202020290a'
        ,
    'src/rl/gym_env.py':
        '...'
        ,
}
for relpath, hexstr in _files.items():
    p = Path(REPO) / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(bytes.fromhex(hexstr))
    print(f'  written: {relpath} ({p.stat().st_size:,} bytes)')
print('v2.21c files ready.')

In [ ]:
# Smoke tests + a v2.21 pilot that runs real gradient steps.
# Catches import errors and training-loop bugs before committing ~1 hr of GPU time.
import subprocess, sys
REPO = '/content/thesis'
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short'],cwd=REPO).returncode==0,'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short'],cwd=REPO).returncode==0,'CRITIC TESTS FAILED'
print('\nv2.21 pilot (runs real gradient steps with NStepReplayBufferExact)...')
from src.rl import configs_v221c
import copy
_pilot = copy.deepcopy(configs_v221c.CONFIGS['A'])
_pilot['learning_starts'] = 200  # small enough for 1200-step pilot
configs_v221c.CONFIGS['PILOT'] = _pilot
from src.rl.train_v221c_td3 import train_td3_v221c
m = train_td3_v221c('PILOT', seed=999, output_dir='/content/pilot', total_timesteps=1200)
import glob, zipfile, io, torch
ck = glob.glob('/content/pilot/td3_v221c_termyield_seed999/*final*.zip')
if ck:
    with zipfile.ZipFile(ck[0]) as z:
        sd = torch.load(io.BytesIO(z.open('policy.pth').read()), map_location='cpu', weights_only=False)
    assert 'actor.log_std.weight' not in sd, 'BUG: log_std found (SAC checkpoint loaded?)'
    assert 'actor.mu_head.weight' in sd, 'BUG: mu_head missing'
    assert abs(float(sd['actor.obs_norm_marker'].item()) - 2.19) < 0.01, 'BUG: marker != 2.19'
    print('  Checkpoint: deterministic actor, mu_head present, marker=2.19. OK')
assert type(m.replay_buffer).__name__ == 'NStepReplayBufferExact', 'BUG: wrong buffer type'
assert m.replay_buffer.n_steps == 5, 'BUG: wrong n_steps'
print(f'  Buffer: {type(m.replay_buffer).__name__}  n_steps={m.replay_buffer.n_steps}  gamma={m.replay_buffer._n_gamma}. OK')
print(f'  model.gamma = {m.gamma:.6f}  (expect {0.99**5:.6f}). {"OK" if abs(m.gamma - 0.99**5)<1e-8 else "BUG"}')
print('\nOK pre-flight passed. Proceed to training.')


In [ ]:
# Full 250k TD3 v2.21c (additive terminal-yield on the v2.20 Run A increment reward).
# Keeps the full dense r1; adds alpha_T*x4_final/X4_REF once at episode end. ~1-1.5 hr T4.
SEED = 0    # CHANGE per session
REPO = '/content/thesis'
from src.rl.train_v221c_td3 import train_td3_v221c
model = train_td3_v221c(
    config_name='A',
    seed=SEED,
    output_dir=f'{REPO}/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    # ── params from CONFIGS['A'] in configs_v221c.py ──
    # INHERITED FROM v2.20 RUN A (unchanged): n_steps=5, gamma_base=0.99,
    #   model.gamma=0.99**5, learning_starts=50_000, reward_du_alpha=0.005, warmup off.
    # SCALES reverted in gym_env.py: X4_REF=600, ALPHA2=0.016 (v2.20 Run A values).
    # THE ONLY v2.21c CHANGE:
    #   biomass_shaping=False      -> KEEP the dense increment r1 (NOT gamma-shaped)
    #   reward_terminal_yield=1.0  -> ADD alpha_T*x4_final/X4_REF once at episode end
    #                                 (season-sum calibrated; tune [0.5,1.5]).
)
print('Training complete.')

In [ ]:
import shutil, os, datetime
RUN_LABEL = 'termyield'   # 'nstep5_damped' for Run B
src=f'/content/thesis/results/rl/td3_v221c_{RUN_LABEL}_seed{SEED}'
dst=os.path.join(DRIVE_ROOT,f'td3_v221c_{RUN_LABEL}_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:',dst)


In [ ]:
# Post-training eval: best_model + final model, perfect + noisy forecasts.
# Writes parquets to results/runs/<tag>/ for the comparison diagnostic.
import subprocess, sys, os
REPO = '/content/thesis'
RUN_LABEL = 'termyield'   # match what you ran
run_dir = f'{REPO}/results/rl/td3_v221c_{RUN_LABEL}_seed{SEED}'
model_path = f'{run_dir}/best_model/best_model.zip'
final_path = f'{run_dir}/td3_v221c_{RUN_LABEL}_seed{SEED}_final.zip'
BEST_TAG=f'td3_v221c_{RUN_LABEL}_best_seed{SEED}'; FINAL_TAG=f'td3_v221c_{RUN_LABEL}_final_seed{SEED}'
print('Evaluating BEST (perfect)...')
r=subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','perfect',
    '--force','--out-tag',BEST_TAG],capture_output=True,text=True,cwd=REPO)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:',r.stderr[-2500:])
assert r.returncode==0,'PERFECT EVAL FAILED'
print('\nEvaluating BEST (noisy, forecast-sensitivity check)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','noisy',
    '--noise-seed','42','--force','--out-tag',BEST_TAG],cwd=REPO)
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k, perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
        '--model',final_path,'--scenario','all','--budget','all','--forecast','perfect',
        '--force','--out-tag',FINAL_TAG],cwd=REPO)
print(f'\nBest-model eval -> results/runs/{BEST_TAG}/')


In [ ]:
# PRIMARY v2.21c DIAGNOSTIC: did the additive terminal term cut the drought seesaw
# WITHOUT losing yield or the waterlog advantage? (vs MPC and the v2.20 Run A baseline)
import pandas as pd, numpy as np, json, glob, os
REPO = '/content/thesis'; RUN_LABEL = 'termyield'
OUT=f'{REPO}/results/runs/td3_v221c_{RUN_LABEL}_best_seed{SEED}'
MPC=f'{REPO}/results/runs'
assert os.path.isdir(OUT), f'{OUT} missing -- run the eval cell first.'
def load_m(d, scen, b, kind):
    g = (glob.glob(os.path.join(d,f'sac_perfect_det_{scen}_rice_{b}pct_seed*.json')) if kind=='v221c'
         else glob.glob(os.path.join(d,f'mpc_perfect_{scen}_rice_{b}pct_Hp8.json')))
    return json.load(open(g[0]))['final_metrics'] if g else None
print('='*80)
print(f'{"cell":<13}{"v221c Y":>8}{"MPC Y":>8}{"d":>6} | {"v221c dr":>9}{"MPC dr":>8} | {"v221c wlog":>11}')
yA,yM,dA,dM,wA,wM=[],[],[],[],[],[]
for scen in ['dry','moderate','wet']:
    for b in ['100','85','70']:
        a=load_m(OUT,scen,b,'v221c'); m=load_m(MPC,scen,b,'mpc')
        if a and m:
            yA.append(a['yield_kg_ha']); yM.append(m['yield_kg_ha'])
            dA.append(a['drought_days_per_agent']); dM.append(m['drought_days_per_agent'])
            wA.append(a['waterlog_days_per_agent']); wM.append(m['waterlog_days_per_agent'])
            print(f'{scen+"/"+b+"%":<13}{a["yield_kg_ha"]:>8.0f}{m["yield_kg_ha"]:>8.0f}'
                  f'{a["yield_kg_ha"]-m["yield_kg_ha"]:>+6.0f} | '
                  f'{a["drought_days_per_agent"]:>9.1f}{m["drought_days_per_agent"]:>8.1f} | '
                  f'{a["waterlog_days_per_agent"]:>11.1f}')
print('='*80)
ay,my=np.mean(yA),np.mean(yM); ad,md=np.mean(dA),np.mean(dM); aw,mw=np.mean(wA),np.mean(wM)
print(f'9-CELL MEAN  yield: v221c={ay:.0f} ({100*ay/my:.1f}% MPC)   drought-d: {ad:.1f} (MPC {md:.1f})   waterlog-d: {aw:.1f} (MPC {mw:.1f})')
print(f'  REFERENCE  v2.20 Run A:  yield 3802 (99.8%)   drought 33.1   waterlog 6.5')
mod70=load_m(OUT,'moderate','70','v221c'); mmod=load_m(MPC,'moderate','70','mpc')
print(f'  mod/70%:  yield v221c={mod70["yield_kg_ha"]:.0f} vs MPC {mmod["yield_kg_ha"]:.0f} (v2.20 RunA -150; gamma-shaping -380)')
print('GATE: drought < v2.20 (33) toward MPC AND yield >= 99.8% (>=v2.20) AND waterlog <= MPC (7.7).')

In [ ]:
# STABILITY DIAGNOSTIC. q_pred calibration + collapse guard + coverage.
# Stage-1 success = q_pred BOUNDED (not monotone), guard never trips.
import os, glob, numpy as np, pandas as pd
REPO = '/content/thesis'
RUN_LABEL = 'termyield'
run_dir=f'{REPO}/results/rl/td3_v221c_{RUN_LABEL}_seed{SEED}'
br=os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    b=pd.read_csv(br); print('--- bias_ratio_log (q_pred trajectory) ---')
    print(b.to_string(index=False))
    q_min=float(b['q_pred_mean'].min()); q_final=float(b['q_pred_mean'].iloc[-1])
    q_mono = all(b['q_pred_mean'].diff().dropna() < 0)   # True if always decreasing
    print(f'\n  q_pred min={q_min:+.1f}  final={q_final:+.1f}  monotone_dive={q_mono}')
    if q_mono:
        verdict = 'FAIL -- monotone dive (same as v2.20 r5). Try Run B.'
    elif q_min < -30 and q_final >= -5:
        verdict = 'PARTIAL -- deep dip but recovered. Watch over seeds; consider Run B.'
    elif q_final >= -5:
        verdict = 'PASS -- q_pred bounded and recovered.'
    else:
        verdict = f'UNCERTAIN -- q_pred final={q_final:+.1f}. Check the curve.'
    print(f'  VERDICT: {verdict}')
else:
    print('No bias_ratio_log.csv at', br)

cg=os.path.join(run_dir,'collapse_guard_log.csv')
if os.path.exists(cg):
    g=pd.read_csv(cg)
    tripped=int(g['collapsed'].max()) if 'collapsed' in g.columns and len(g) else 0
    last_frac=g['frac_low_rolling'].iloc[-1] if len(g) else float('nan')
    print(f'\n--- collapse_guard ---  rows={len(g)}  final_frac_low={last_frac:.0%}')
    print(f'  guard tripped: {"YES -- collapsed" if tripped else "NO -- healthy"}')

cov=os.path.join(run_dir,'low_action_coverage_log.csv')
if os.path.exists(cov):
    c=pd.read_csv(cov)
    print(f'--- low_action_coverage ---  mean frac_low (last 50k)={c["frac_low_action"].tail(50).mean():.0%}')

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    runs=glob.glob(os.path.join(run_dir,'tensorboard','*'))
    if runs:
        ea=EventAccumulator(runs[0]); ea.Reload()
        if 'train/critic_loss' in ea.Tags()['scalars']:
            cl=ea.Scalars('train/critic_loss'); mxl=max(e.value for e in cl)
            print(f'\n  max critic_loss = {mxl:.2f}  (STABLE if < 100; v2.7 cascade hit 6.9e12)')
except Exception as e:
    print('tensorboard read skipped:', e)


## [NEXT VERSION] Pulsing / Markov-r5 (needs a network change)

v2.21 targets **drought** via the gamma-correct biomass objective. The other gap — **pulsing** (mean|Δu| 2.04 vs MPC 0.98) — needs r5 made *Markov* (`expose_prev_u=True`), which requires a 9-feature actor+critic first (`networks_td3.py` hard-codes 8). Make it its own version (v2.22).

In [ ]:
# [OPTIONAL] Resume v2.21 from a checkpoint (after a Colab disconnect).
# from src.rl.train_v221c_td3 import WarmupAsymmetricLRTD3
# SEED=0; RUN_LABEL='termyield'; STEP=150_000
# ckpt=f'/content/thesis/results/rl/td3_v221c_{RUN_LABEL}_seed{SEED}/checkpoints/td3_v221c_{RUN_LABEL}_seed{SEED}_{STEP}_steps.zip'